### **Importing libraries**

In [20]:
import numpy as np
import pandas as pd
import GEOparse

from scipy.stats import ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import SelectKBest, f_classif

### **Load dataset from GEO.soft file**
Replace the path with your actual file location

In [ ]:
gse = GEOparse.get_GEO(filepath=r"C:\Users\Asrefanavary\Desktop\GSE18123_family.soft")

23-Jul-2026 16:57:46 INFO GEOparse - Parsing C:\Users\Asrefanavary\Desktop\GSE18123_family.soft: 
23-Jul-2026 16:57:46 DEBUG GEOparse - DATABASE: GeoMiame
23-Jul-2026 16:57:46 DEBUG GEOparse - SERIES: GSE18123
23-Jul-2026 16:57:46 DEBUG GEOparse - PLATFORM: GPL570
c:\Users\Asrefanavary\anaconda3\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
23-Jul-2026 16:57:49 DEBUG GEOparse - PLATFORM: GPL6244
23-Jul-2026 16:57:53 DEBUG GEOparse - SAMPLE: GSM650510
23-Jul-2026 16:57:53 DEBUG GEOparse - SAMPLE: GSM650512
23-Jul-2026 16:57:53 DEBUG GEOparse - SAMPLE: GSM650513
23-Jul-2026 16:57:54 DEBUG GEOparse - SAMPLE: GSM650514
23-Jul-2026 16:57:54 DEBUG GEOparse - SAMPLE: GSM650515
23-Jul-2026 16:57:54 DEBUG GEOparse - SAMPLE: GSM650516
23-Jul-2026 16:57:54 DEBUG GEOparse - SAMPLE: GSM650517
23-Jul-2026 16:57:54 DEBUG GEOparse - SAMPLE: GSM65


### **Inspecting the GEO Sample**

The first GEO sample (GSM) is extracted from the downloaded dataset, and its expression matrix is examined. The dataset contains 54,613 probes, with each row representing a probe identifier (ID_REF) and its corresponding normalized expression value (VALUE).

In [22]:
gsm = list(gse.gsms.values())[0]

print(gsm.table.shape)
gsm.table.head()

(54613, 2)


,ID_REF,VALUE
0,1007_s_at,26.65640
1,1053_at,39.00726
2,117_at,296.01724
3,121_at,209.03575
4,1255_g_at,63.82730


### **Extracting Sample Metadata**

The metadata of each GEO sample (GSM) are extracted to build a structured sample information table. For every sample, the diagnosis, gender, age at blood collection, and microarray platform are retrieved from the available metadata and stored in a pandas DataFrame.

The resulting table contains **285 samples** and **5 metadata fields** (`Sample`, `Diagnosis`, `Gender`, `Age`, and `Platform`), providing the clinical and demographic information required for subsequent data preprocessing and analysis.


In [23]:
samples_info = []

for gsm_id, gsm in gse.gsms.items():

    diagnosis = None
    gender = None
    age = None

    for item in gsm.metadata.get("characteristics_ch1", []):

        if item.startswith("diagnosis:"):
            diagnosis = item.replace("diagnosis:", "").strip()

        elif item.startswith("gender:"):
            gender = item.replace("gender:", "").strip()

        elif item.startswith("age at blood drawing"):
            age = int(item.split(":")[-1].strip())

    samples_info.append({
        "Sample": gsm_id,
        "Diagnosis": diagnosis,
        "Gender": gender,
        "Age": age,
        "Platform": gsm.metadata["platform_id"][0]
    })

df_samples = pd.DataFrame(samples_info)

In [24]:
df_samples

,Sample,Diagnosis,Gender,Age,Platform
0,GSM650510,PDD-NOS,male,118,GPL570
1,GSM650512,PDD-NOS,male,79,GPL570
2,GSM650513,AUTISM,male,169,GPL570
3,GSM650514,AUTISM,male,92,GPL570
4,GSM650515,AUTISM,male,78,GPL570
...,...,...,...,...,...
280,GSM650959,CONTROL,male,28,GPL6244
281,GSM650968,CONTROL,male,51,GPL6244
282,GSM650969,CONTROL,male,37,GPL6244
283,GSM650970,CONTROL,male,49,GPL6244


### **Selecting the GPL6244 Platform**

The dataset contains samples generated using two different microarray platforms: **GPL570 (99 samples)** and **GPL6244 (186 samples)**. 
Since **GPL6244** includes a substantially larger number of samples, it was selected for the subsequent analyses. Using the platform with the larger sample size provides greater statistical power and ensures a more comprehensive analysis while avoiding potential technical variability introduced by combining data from different platforms.

In [56]:
df_samples["Platform"].unique()

array(['GPL570', 'GPL6244'], dtype=object)

In [57]:
df_samples["Platform"].value_counts()

Platform
GPL6244    186
GPL570      99
Name: count, dtype: int64

In [58]:
df_gpl6244 = df_samples[df_samples["Platform"] == "GPL6244"].copy()

df_gpl6244.shape

(186, 5)

In [26]:
df_gpl6244.head(10)

,Sample,Diagnosis,Gender,Age,Platform
99,GSM650655,ASPERGER'S DISORDER,male,156,GPL6244
100,GSM650656,AUTISM,male,168,GPL6244
101,GSM650657,CONTROL,male,228,GPL6244
102,GSM650658,AUTISM,male,60,GPL6244
103,GSM650659,AUTISM,male,36,GPL6244
104,GSM650660,PDD-NOS,male,72,GPL6244
105,GSM650661,PDD-NOS,male,108,GPL6244
106,GSM650662,ASPERGER'S DISORDER,male,120,GPL6244
107,GSM650663,PDD-NOS,male,48,GPL6244
108,GSM650664,PDD-NOS,male,108,GPL6244


### **Diagnostic Grouping Strategy**

The original dataset contained four diagnostic categories: **CONTROL**, **PDD-NOS**, **AUTISM**, and **ASPERGER'S DISORDER**. According to the **DSM-IV** criteria, Autism, Asperger's Disorder, and PDD-NOS were classified as pervasive developmental disorders and represent related neurodevelopmental conditions within the autism spectrum.

In the updated **DSM-5** framework, these previously separate diagnoses were integrated into a single clinical category known as **Autism Spectrum Disorder (ASD)**.

The original dataset contained four diagnostic categories:

| Diagnosis | Number of Samples |
|------------|------------------|
| CONTROL | 82 |
| PDD-NOS | 48 |
| AUTISM | 41 |
| ASPERGER'S DISORDER | 15 |

Since some diagnostic subgroups contained a limited number of samples, especially **Asperger's Disorder**, performing a multi-class classification task could decrease statistical power and increase the possibility of model overfitting. Therefore, all ASD-related diagnoses (**AUTISM**, **PDD-NOS**, and **ASPERGER'S DISORDER**) were combined into one unified ASD class, while **CONTROL** samples were assigned to the healthy control class.

This conversion resulted in a binary classification problem:

- **Control (Label = 0):** Healthy individuals  
- **ASD (Label = 1):** Individuals with autism spectrum-related diagnoses  

This strategy enables a more robust classification model by increasing the number of samples in the disease group and focusing on the primary objective of ASD detection rather than distinguishing between different ASD subtypes.

In [28]:
def convert_label(diagnosis):
    if "CONTROL" in diagnosis:
        return 0
    else:
        return 1


df_gpl6244["Label"] = df_gpl6244["Diagnosis"].apply(convert_label)

df_gpl6244.head()

,Sample,Diagnosis,Gender,Age,Platform,Label
99,GSM650655,ASPERGER'S DISORDER,male,156,GPL6244,1
100,GSM650656,AUTISM,male,168,GPL6244,1
101,GSM650657,CONTROL,male,228,GPL6244,0
102,GSM650658,AUTISM,male,60,GPL6244,1
103,GSM650659,AUTISM,male,36,GPL6244,1


In [29]:
df_gpl6244["Label"].value_counts()

Label
1    104
0     82
Name: count, dtype: int64

In [31]:
df_gpl6244["Diagnosis"].value_counts()

Diagnosis
CONTROL                82
PDD-NOS                48
AUTISM                 41
ASPERGER'S DISORDER    15
Name: count, dtype: int64

### **Gender Distribution Analysis**

The gender distribution of the selected GPL6244 dataset was examined to evaluate the demographic composition of the samples. The dataset contains **128 male samples** and **58 female samples**.

The distribution across diagnostic groups shows that the control group (**Label = 0**) includes **48 males** and **34 females**, while the ASD group (**Label = 1**) contains **80 males** and **24 females**. This analysis highlights the gender imbalance commonly observed in ASD datasets, where males are more frequently represented than females.

In [27]:
print(df_gpl6244['Gender'].value_counts())

Gender
male      128
female     58
Name: count, dtype: int64


In [30]:
print(df_gpl6244.groupby(['Label', 'Gender']).size())

Label  Gender
0      female    34
       male      48
1      female    24
       male      80
dtype: int64


### **Initial Statistical Description**

The dataset contains 186 samples with complete metadata and no missing values.  
The samples were obtained from the GPL6244 platform. The dataset includes 128 male and 58 female participants.  

The age of participants ranged from 24 to 264 months, with a mean age of 96.8 months.  
The median age was 84 months, indicating a wide age distribution among samples.

All samples were generated using the same microarray platform, ensuring consistency across the dataset.

In [59]:
df_gpl6244.isnull().sum()

Sample       0
Diagnosis    0
Gender       0
Age          0
Platform     0
dtype: int64

In [1]:
df_gpl6244.describe(include="all")

NameError: name 'df_gpl6244' is not defined

Data Exploration

 - Load GEO Dataset ✅

 - Inspect Dataset Structure ✅

 - Extract Sample Metadata ✅

 - Check Platforms ✅

 - Select GPL6244 Platform ✅

 - Convert Diagnosis to Binary Labels ✅

 - Analyze Diagnosis Distribution ✅

 - Analyze Gender Distribution ✅

 - Missing Value Analysis ✅

 - Initial Statistical Description ✅